In [ ]:
# Imports básicos
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import time

# Configuración del modelo pequeño para iterar rápido
llm_small = ChatOllama(
    model="qwen2.5:3b-instruct",
    temperature=0,  # Determinista para debugging
)

print("✓ Imports OK")
print(f"✓ Modelo configurado: {llm_small.model}")

In [ ]:
start = time.time()

response = llm_small.invoke("Reply with exactly one word: OK")

elapsed = time.time() - start
print(f"Respuesta: {response.content}")
print(f"Tiempo: {elapsed:.2f}s")

In [ ]:
# Manera 1: mensaje directo (menos control)
response1 = llm_small.invoke("Translate 'Hello' to Spanish.")
print("Sin system:")
print(response1.content)
print("---")

# Manera 2: con system prompt (más control)
messages = [
    SystemMessage(content="You are a translator. Reply ONLY with the translation, no extra words."),
    HumanMessage(content="Translate 'Hello' to Spanish.")
]
response2 = llm_small.invoke(messages)
print("Con system:")
print(response2.content)

In [ ]:
messages = [
    SystemMessage(content="""You are a tech recruiter assistant. Given a skill name, 
provide a brief 1-sentence description of what it is and its typical use case."""),
    HumanMessage(content="MLflow")
]

response = llm_small.invoke(messages)
print(response.content)

In [ ]:
import pdfplumber
from pathlib import Path

cv_path = Path("./data/AlexiaHerrador_IA.pdf")
print(f"¿Existe el CV? {cv_path.exists()}")
print(f"Ruta absoluta: {cv_path.resolve()}")

In [ ]:
with pdfplumber.open(cv_path) as pdf:
    cv_text = "\n".join(page.extract_text() or "" for page in pdf.pages)

print(f"Longitud del texto: {len(cv_text)} caracteres")
print(f"Primeras 500 chars:\n{cv_text[:500]}")
print(f"\n---\n\nÚltimas 300 chars:\n{cv_text[-300:]}")

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Experience(BaseModel):
    """Una experiencia profesional."""
    company: str = Field(description="Company name")
    role: str = Field(description="Job title or role")
    start_date: str = Field(description="Format: YYYY-MM or YYYY")
    end_date: Optional[str] = Field(
        default=None,
        description="Format: YYYY-MM, YYYY, or 'current' if ongoing"
    )
    description: str = Field(description="Short summary of responsibilities and achievements")

class Education(BaseModel):
    """Formación académica."""
    institution: str
    degree: str = Field(description="Degree name, e.g. 'Grado Superior en DAM'")
    field: Optional[str] = Field(default=None, description="Field of study if different from degree")
    year: Optional[str] = Field(default=None, description="Year of completion or range")

class CVProfile(BaseModel):
    """Perfil completo extraído del CV."""
    full_name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    summary: str = Field(description="2-3 sentence professional summary based on the CV")
    skills: list[str] = Field(description="Technical skills only, normalized (e.g. 'Python' not 'python programming')")
    languages: list[str] = Field(description="Spoken languages with level if mentioned")
    experiences: list[Experience]
    education: list[Education]
    target_roles: list[str] = Field(
        description="3-5 inferred target job titles this person could realistically apply to, based on their profile. Be specific (e.g. 'Junior MLOps Engineer', not just 'Engineer')."
    )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import time

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a CV parsing expert. Extract structured information from the CV text below.

Rules:
- Only extract information explicitly present in the CV, except for 'target_roles' which you MUST infer from skills and experience.
- For 'target_roles', suggest 3-5 specific job titles (e.g., 'Junior MLOps Engineer', 'Data Engineer', not generic ones like 'Engineer').
- For dates, normalize to YYYY-MM format when possible (e.g., 'Septiembre 2023' → '2023-09').
- For ongoing roles, use 'current' as end_date.
- For skills, extract individual items (e.g., 'Python', 'SQL' as separate skills).
- For languages, include the level if mentioned (e.g., 'Spanish (Native)', 'German (Basic)').
- If a field is not present, use null for optional fields or empty list for lists.
- Return ONLY valid JSON matching the schema. No explanations or markdown."""),
    ("human", "CV text:\n\n{cv_text}")
])

# Creamos la cadena: prompt → LLM con structured output
structured_llm = llm_small.with_structured_output(CVProfile)
chain = extraction_prompt | structured_llm

# Ejecutamos
print("Extrayendo CV... esto puede tardar 30-60s con el modelo 3B en CPU.\n")
start = time.time()
profile = chain.invoke({"cv_text": cv_text})
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(profile.model_dump_json(indent=2))

In [ ]:
# Modelo grande para ejecución real
llm_large = ChatOllama(
    model="qwen2.5:7b-instruct",
    temperature=0,
)

print(f"✓ Modelo grande configurado: {llm_large.model}")

In [ ]:
import time

start = time.time()
response = llm_large.invoke("Reply with exactly one word: OK")
elapsed = time.time() - start

print(f"Respuesta: {response.content}")
print(f"Tiempo (incluye carga del modelo en RAM): {elapsed:.1f}s")

In [ ]:
# Misma chain, pero con el modelo grande
structured_llm_large = llm_large.with_structured_output(CVProfile)
chain_large = extraction_prompt | structured_llm_large

print("Extrayendo CV con 7B... esto puede tardar 3-5 minutos con el modelo 7B en CPU.\n")
start = time.time()
profile_large = chain_large.invoke({"cv_text": cv_text})
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(profile_large.model_dump_json(indent=2))

In [ ]:
def load_cv_text(path: str | Path) -> str:
    """
    Load CV text from various formats.
    
    Supports: .pdf, .md, .txt
    """
    path = Path(path)
    
    if not path.exists():
        raise FileNotFoundError(f"CV not found: {path}")
    
    suffix = path.suffix.lower()
    
    if suffix == ".pdf":
        with pdfplumber.open(path) as pdf:
            return "\n".join(page.extract_text() or "" for page in pdf.pages)
    
    elif suffix in {".md", ".txt"}:
        return path.read_text(encoding="utf-8")
    
    else:
        raise ValueError(f"Unsupported CV format: {suffix}. Use .pdf, .md, or .txt")


# Test con el fixture sintético
fixture_path = Path("./tests/fixtures/cv_junior_tech.md")
fixture_text = load_cv_text(fixture_path)

print(f"Longitud: {len(fixture_text)} caracteres")
print(f"Primeras 300 chars:\n{fixture_text[:300]}")

In [ ]:
print("Extrayendo CV sintético con Qwen 2.5 7B... 3-5 min con CPU.\n")
start = time.time()
fixture_profile = chain_large.invoke({"cv_text": fixture_text})
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(fixture_profile.model_dump_json(indent=2))

In [ ]:
import re
from typing import NamedTuple

class ContactInfo(NamedTuple):
    email: str | None
    phone: str | None
    linkedin: str | None
    github: str | None

def extract_contact_info(text: str) -> ContactInfo:
    """
    Extract deterministic contact fields using regex.
    Much more reliable than LLM for structured data.
    """
    # Email: standard RFC-ish pattern
    email_match = re.search(
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        text,
    )
    
    # Phone: international format with optional country code
    # Handles: +34 657 641 482, +34657641482, 657641482, 657-641-482
    phone_match = re.search(
        r"(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3}[\s-]?\d{3}[\s-]?\d{3,4}",
        text,
    )
    
    # LinkedIn URL or handle
    linkedin_match = re.search(
        r"linkedin\.com/in/[\w-]+|LinkedIn:\s*([\w\s-]+)",
        text,
        re.IGNORECASE,
    )
    
    # GitHub URL or handle
    github_match = re.search(
        r"github\.com/[\w-]+",
        text,
        re.IGNORECASE,
    )
    
    return ContactInfo(
        email=email_match.group(0) if email_match else None,
        phone=phone_match.group(0).strip() if phone_match else None,
        linkedin=linkedin_match.group(0) if linkedin_match else None,
        github=github_match.group(0) if github_match else None,
    )


# Probamos con ambos CVs
print("=== Tu CV real ===")
print(extract_contact_info(cv_text))

print("\n=== Fixture sintético ===")
print(extract_contact_info(fixture_text))

In [ ]:
# Nuevo schema: sacamos email/phone del input del LLM (los pone regex)
class CVProfileSemantic(BaseModel):
    """CV profile for the LLM to extract (semantic fields only)."""
    full_name: str
    location: Optional[str] = None
    summary: str = Field(description="2-3 sentence professional summary based on the CV")
    skills: list[str] = Field(description="Individual technical skills, normalized. Include ALL technologies, tools, and frameworks mentioned anywhere in the CV (including experience descriptions).")
    languages: list[str] = Field(description="Languages with level (e.g. 'English (B2)', 'German (Basic)')")
    experiences: list[Experience]
    education: list[Education]
    target_roles: list[str] = Field(
        description="3-5 specific Junior-level job titles this person should apply to. EVERY role MUST start with 'Junior' (e.g., 'Junior MLOps Engineer', 'Junior Data Engineer'). Base them on the candidate's actual skills and experience, not generic guesses."
    )

# Nuevo prompt con instrucciones más fuertes
extraction_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", """You extract structured information from CVs.

CRITICAL RULES:
1. Only extract info explicitly in the CV, except 'target_roles' (you infer).
2. For 'target_roles': EVERY role MUST start with 'Junior'. Be specific. Base on actual skills, not guesses.
3. Skills: extract from EVERYWHERE (skills section AND experience descriptions). Don't miss technologies mentioned in job descriptions.
4. Dates: YYYY-MM format (e.g., 'Septiembre 2023' → '2023-09'). Ongoing → 'current'.
5. Languages: include level ('Native', 'Fluent', 'B2', 'Basic', etc.)
6. If info not present, use null for optional fields or empty list.
7. Return ONLY valid JSON. No markdown. No explanations."""),
    ("human", "CV text:\n\n{cv_text}")
])

structured_llm_v2 = llm_large.with_structured_output(CVProfileSemantic)
chain_v2 = extraction_prompt_v2 | structured_llm_v2

In [ ]:
from pydantic import BaseModel

class CVProfileFull(BaseModel):
    """Complete CV profile combining regex-extracted and LLM-extracted fields."""
    # From regex
    email: Optional[str] = None
    phone: Optional[str] = None
    linkedin: Optional[str] = None
    github: Optional[str] = None
    # From LLM
    full_name: str
    location: Optional[str] = None
    summary: str
    skills: list[str]
    languages: list[str]
    experiences: list[Experience]
    education: list[Education]
    target_roles: list[str]


def extract_full_profile(cv_text: str) -> CVProfileFull:
    """
    Full CV extraction: regex for contact, LLM for semantic fields.
    """
    # Step 1: regex (instant)
    contact = extract_contact_info(cv_text)
    
    # Step 2: LLM (slow but smart)
    semantic = chain_v2.invoke({"cv_text": cv_text})
    
    # Step 3: merge
    return CVProfileFull(
        email=contact.email,
        phone=contact.phone,
        linkedin=contact.linkedin,
        github=contact.github,
        **semantic.model_dump()
    )


# Probamos con fixture (porque es más rápido iterar)
print("Extrayendo fixture con pipeline completo (regex + LLM)...")
start = time.time()
full_profile = extract_full_profile(fixture_text)
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(full_profile.model_dump_json(indent=2))

In [ ]:
def clean_linkedin(raw: str | None) -> str | None:
    """Clean LinkedIn match to get just the handle or URL."""
    if not raw:
        return None
    # Remove 'LinkedIn:' prefix, whitespace, newlines
    cleaned = re.sub(r"^LinkedIn:\s*", "", raw, flags=re.IGNORECASE)
    cleaned = cleaned.strip()
    # If it has whitespace in the middle, it's probably a name, not a URL
    # Keep just URL if it's a URL, otherwise keep the name
    return cleaned if cleaned else None


def extract_contact_info_v2(text: str) -> ContactInfo:
    """Improved contact extraction with cleaning."""
    email_match = re.search(
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        text,
    )
    
    phone_match = re.search(
        r"(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3}[\s-]?\d{3}[\s-]?\d{3,4}",
        text,
    )
    
    linkedin_match = re.search(
        r"linkedin\.com/in/[\w-]+|LinkedIn:\s*([\w\s-]+?)(?=\s*\||\s*$|\s*\n)",
        text,
        re.IGNORECASE,
    )
    
    github_match = re.search(
        r"github\.com/[\w-]+",
        text,
        re.IGNORECASE,
    )
    
    return ContactInfo(
        email=email_match.group(0) if email_match else None,
        phone=phone_match.group(0).strip() if phone_match else None,
        linkedin=clean_linkedin(linkedin_match.group(0)) if linkedin_match else None,
        github=github_match.group(0) if github_match else None,
    )


# Test rápido sin llamar al LLM
print("Tu CV real:")
print(extract_contact_info_v2(cv_text))
print("\nFixture:")
print(extract_contact_info_v2(fixture_text))

In [ ]:
extraction_prompt_v3 = ChatPromptTemplate.from_messages([
    ("system", """You extract structured information from CVs. You are thorough and never miss information.

CRITICAL RULES:
1. Only extract info explicitly in the CV, EXCEPT 'target_roles' (you MUST infer).
2. 'target_roles': return EXACTLY 3-5 roles. EVERY role starts with 'Junior'. Be specific based on skills shown.
3. 'skills': extract EVERY technology, framework, language, tool, database, platform mentioned ANYWHERE in the CV. 
   - Look in the skills section AND in experience descriptions.
   - Common misses to avoid: SQL, JavaScript, REST APIs, Flask, pytest.
   - Normalize naming (e.g., 'python' → 'Python', 'rest apis' → 'REST APIs').
4. 'location': if any city/country is mentioned near the name or contact info, extract it. Don't return null if location text is present.
5. 'languages': include level ('Native', 'Fluent', 'B2', 'Basic', etc.).
6. Dates: YYYY-MM format ('Septiembre 2023' → '2023-09'). Ongoing → 'current'.
7. For optional fields not present, use null. For missing lists, use empty list.
8. Return ONLY valid JSON. No markdown. No explanations.

Example of GOOD 'target_roles' for a junior with Python + Airflow + SQL:
["Junior Data Engineer", "Junior Python Developer", "Junior ETL Engineer", "Junior Analytics Engineer"]

Example of BAD 'target_roles' (too generic, missing 'Junior'):
["Data Engineer", "Developer", "Engineer"]"""),
    ("human", "CV text:\n\n{cv_text}")
])

structured_llm_v3 = llm_large.with_structured_output(CVProfileSemantic)
chain_v3 = extraction_prompt_v3 | structured_llm_v3

# Actualizamos la función wrapper
def extract_full_profile_v2(cv_text: str) -> CVProfileFull:
    contact = extract_contact_info_v2(cv_text)
    semantic = chain_v3.invoke({"cv_text": cv_text})
    return CVProfileFull(
        email=contact.email,
        phone=contact.phone,
        linkedin=contact.linkedin,
        github=contact.github,
        **semantic.model_dump()
    )

In [ ]:
print("Extracción v2 con pipeline completo (regex v2 + LLM v3)...")
start = time.time()
full_profile_v2 = extract_full_profile_v2(fixture_text)
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(full_profile_v2.model_dump_json(indent=2))

In [ ]:
def post_process_experiences(experiences: list[Experience], cv_text: str) -> list[Experience]:
    """
    Fix common LLM mistakes on experience end_dates.
    
    Strategy:
    - If end_date is null AND the description or nearby text mentions ongoing keywords,
      set it to 'current'.
    - This is a deterministic fallback; the LLM sometimes omits end_date entirely.
    """
    ongoing_keywords = [
        "actualmente", "actualidad", "presente", "actual",
        "present", "current", "currently", "ongoing",
        "hasta la fecha", "hasta hoy", "now"
    ]
    
    cv_lower = cv_text.lower()
    
    fixed = []
    for exp in experiences:
        if exp.end_date is None:
            # Check if the role/company appears near an 'ongoing' keyword in the CV
            role_lower = exp.role.lower()
            company_lower = exp.company.lower()
            
            # Find the position of the role or company in the CV
            anchor_pos = cv_lower.find(role_lower)
            if anchor_pos == -1:
                anchor_pos = cv_lower.find(company_lower)
            
            if anchor_pos != -1:
                # Look at the 200 chars after the anchor (where dates usually are)
                window = cv_lower[anchor_pos:anchor_pos + 200]
                if any(kw in window for kw in ongoing_keywords):
                    exp = exp.model_copy(update={"end_date": "current"})
        
        fixed.append(exp)
    
    return fixed


# Aplicamos post-procesado al último perfil extraído
full_profile_v2.experiences = post_process_experiences(
    full_profile_v2.experiences,
    fixture_text
)

print("Después del post-procesado:\n")
for exp in full_profile_v2.experiences:
    print(f"  {exp.role} @ {exp.company}")
    print(f"    {exp.start_date} → {exp.end_date}\n")

In [ ]:
# Reload forzoso por si el notebook tiene versión vieja en caché
import importlib
import hansel.cv
importlib.reload(hansel.cv)

from hansel.cv import CVExtractor

# Crear el extractor (reutilizable)
extractor = CVExtractor()

# Probar con el fixture
print("Extrayendo con el módulo empaquetado...\n")
start = time.time()
profile = extractor.extract_from_file("./tests/fixtures/cv_junior_tech.md")
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
print(profile.model_dump_json(indent=2))

In [ ]:
# Debug: ¿por qué no encuentra "Present"?
fixture_text_lower = fixture_text.lower()

# Buscar la anchor (rol "Backend Developer")
role = "Backend Developer"
anchor = fixture_text_lower.find(role.lower())
print(f"Anchor position para '{role}': {anchor}")

if anchor != -1:
    window = fixture_text_lower[anchor:anchor + 200]
    print(f"\nVentana de 200 chars a partir del anchor:")
    print("---")
    print(window)
    print("---")
    
    # Qué keywords están o no
    keywords = ["actualmente", "actualidad", "presente", "actual",
                "present", "current", "currently", "ongoing",
                "hasta la fecha", "now"]
    for kw in keywords:
        if kw in window:
            print(f"✓ Encontrado: '{kw}'")

In [ ]:
# Recarga el módulo con los cambios
import importlib
import hansel.cv.extractor
importlib.reload(hansel.cv.extractor)
from hansel.cv.extractor import CVExtractor

extractor = CVExtractor()

print("Extrayendo con post-procesado corregido...")
start = time.time()
profile = extractor.extract_from_file("./tests/fixtures/cv_junior_tech.md")
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")

# Mostramos solo las experiencias, que es lo que nos interesa
for exp in profile.experiences:
    print(f"  {exp.role} @ {exp.company}")
    print(f"    {exp.start_date} → {exp.end_date}\n")

In [ ]:
from pydantic import BaseModel, Field


class JobQuery(BaseModel):
    """A single job search query targeting a specific role/location combination."""
    keywords: str = Field(description="Short search keywords (2-5 words). E.g., 'Junior Data Engineer'")
    location: str = Field(description="City, country, or 'remote'. E.g., 'Zurich', 'Switzerland', 'remote'")
    rationale: str = Field(description="One sentence explaining why this query fits the candidate")


class SearchStrategy(BaseModel):
    """A set of diverse queries to maximize coverage."""
    queries: list[JobQuery] = Field(description="3-5 diverse queries. Must vary in role, location, or scope.")


query_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a job search strategist. Given a candidate profile and preferences, 
generate 3-5 DIVERSE search queries to maximize the chances of finding good matches.

Diversity rules:
- Don't just repeat the same title with different wording.
- Mix broader and more specific roles (e.g., 'Junior Backend Developer' AND 'Junior Python Developer').
- Consider adjacent roles the candidate could realistically apply to.
- Vary locations when appropriate (user's preferred location + remote options).
- Keywords should be search-engine friendly: what a person would type on jobs.ch or Adzuna.

Format rules:
- 'keywords': 2-5 words max. No punctuation. No 'and', no commas.
- 'location': single location, or 'remote'. Use standard names ('Switzerland', not 'CH').
- 'rationale': one short sentence, no fluff.

Return ONLY valid JSON."""),
    ("human", """Candidate profile:
Name: {full_name}
Current summary: {summary}
Skills: {skills}
Target roles (inferred): {target_roles}

Preferences:
Primary location: {preferred_location}
Open to remote: {remote_ok}

Generate diverse job search queries.""")
])

query_chain = query_prompt | llm_large.with_structured_output(SearchStrategy)

# Probamos con el perfil ya extraído
start = time.time()
strategy = query_chain.invoke({
    "full_name": profile.full_name,
    "summary": profile.summary,
    "skills": ", ".join(profile.skills),
    "target_roles": ", ".join(profile.target_roles),
    "preferred_location": "Switzerland (Aarau/Zurich area)",
    "remote_ok": "yes, remote Europe also acceptable",
})
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s\n")
for i, q in enumerate(strategy.queries, 1):
    print(f"{i}. {q.keywords}  @  {q.location}")
    print(f"   Why: {q.rationale}\n")

In [ ]:
# Recarga los módulos
import importlib
import hansel.cv
import hansel.cv.extractor
import hansel.cv.schemas
importlib.reload(hansel.cv.schemas)
importlib.reload(hansel.cv.extractor)
importlib.reload(hansel.cv)

from hansel.cv import CVExtractor, Seniority

extractor = CVExtractor()

# Prueba con JUNIOR
print("=== JUNIOR ===")
start = time.time()
profile_junior = extractor.extract_from_file(
    "./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.JUNIOR,
)
print(f"⏱  {time.time() - start:.1f}s")
print(f"Target roles: {profile_junior.target_roles}\n")


In [ ]:
# Prueba con MID (mismo CV) — para verificar que el seniority cambia el output
print("=== MID ===")
start = time.time()
profile_mid = extractor.extract_from_file(
    "./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.MID,
)
print(f"⏱  {time.time() - start:.1f}s")
print(f"Target roles: {profile_mid.target_roles}")

In [ ]:
# Recarga
import importlib
import hansel.search
import hansel.search.query_generator
importlib.reload(hansel.search.query_generator)
importlib.reload(hansel.search)

from hansel.search import QueryGenerator

qg = QueryGenerator()

# Generamos queries con el perfil Junior (ya extraído antes)
print("=== Queries para perfil JUNIOR ===\n")
start = time.time()
strategy = qg.generate(
    profile=profile_junior,
    preferred_location="Switzerland (Aarau/Zurich area)",
    remote_ok=True,
)
print(f"⏱  {time.time() - start:.1f}s\n")

for i, q in enumerate(strategy.queries, 1):
    print(f"{i}. {q.keywords}")
    print(f"   📍 {q.location}")
    print(f"   💡 {q.rationale}\n")

In [ ]:
import os
from dotenv import load_dotenv

# Cargamos el archivo .env (busca automáticamente en el directorio actual y padres)
loaded = load_dotenv()
print(f"¿Se cargó .env? {loaded}")

# Verificamos que las variables están disponibles
adzuna_id = os.getenv("ADZUNA_APP_ID")
adzuna_key = os.getenv("ADZUNA_APP_KEY")
jooble_key = os.getenv("JOOBLE_API_KEY")

# Mostramos solo los primeros/últimos caracteres para confirmar que cargaron
# sin revelar el valor completo por si compartes la salida
def mask(val: str | None) -> str:
    if not val:
        return "❌ NOT SET"
    if len(val) < 8:
        return "⚠️  Too short"
    return f"✓ {val[:4]}...{val[-4:]} ({len(val)} chars)"

print(f"ADZUNA_APP_ID: {mask(adzuna_id)}")
print(f"ADZUNA_APP_KEY: {mask(adzuna_key)}")
print(f"JOOBLE_API_KEY: {mask(jooble_key)}")

In [ ]:
from hansel.sources import JobSourceAdapter, JobListing, JobSource

# Ejemplo ficticio: así luce una JobListing ya normalizada
fake_listing = JobListing(
    source=JobSource.ARBEITNOW,
    source_id="12345",
    url="https://www.arbeitnow.com/jobs/junior-data-engineer-at-acme",
    title="Junior Data Engineer",
    company="ACME Corp",
    location="Remote (Europe)",
    description="We are looking for a Junior Data Engineer...",
    is_remote=True,
    tags=["python", "airflow", "sql"],
)

print("✓ Se puede crear un JobListing válido:")
print(fake_listing.model_dump_json(indent=2))

# Probar la interfaz abstracta: NO se puede instanciar directamente
try:
    bad = JobSourceAdapter()
except TypeError as e:
    print(f"\n✓ La interfaz abstracta no se puede instanciar directamente (esto es lo esperado):")
    print(f"  → {e}")

In [ ]:
import importlib
import hansel.sources
import hansel.sources.arbeitnow
importlib.reload(hansel.sources.arbeitnow)
importlib.reload(hansel.sources)

from hansel.sources import ArbeitnowAdapter

adapter = ArbeitnowAdapter()
print(f"Adapter: {adapter}\n")

# En Jupyter, por ser async, usamos await directamente
listings = await adapter.search(keywords="Python", location=None, limit=5)

print(f"Encontradas {len(listings)} ofertas matching 'Python'\n")
for listing in listings:
    print(f"📌 {listing.title}")
    print(f"   🏢 {listing.company}")
    print(f"   📍 {listing.location} (remote: {listing.is_remote})")
    print(f"   🔗 {listing.url}")
    print(f"   🏷  {listing.tags[:5]}")
    print(f"   📝 {listing.description[:150]}...")
    print()

In [ ]:
# Probamos con keywords más específicas tipo "Junior Data Engineer"
listings_junior = await adapter.search(
    keywords="Junior Developer",
    location=None,
    limit=10
)
print(f"Encontradas {len(listings_junior)} ofertas matching 'Junior Developer'\n")
for l in listings_junior[:3]:
    print(f"📌 {l.title}")
    print(f"   🏢 {l.company}  📍 {l.location}")
    print(f"   Tags: {l.tags}\n")

# Y probemos remote
listings_remote = await adapter.search(
    keywords="Developer",
    location="remote",
    limit=10
)
print(f"\nEncontradas {len(listings_remote)} ofertas REMOTE matching 'Developer'")
for l in listings_remote[:3]:
    print(f"📌 {l.title}  (remote: {l.is_remote})")

In [ ]:
# ¿Cuántas ofertas devuelve Arbeitnow hoy en total?
listings_all = await adapter.search(keywords="", location=None, limit=100)
print(f"Feed completo: {len(listings_all)} ofertas\n")

# Cuántas son remote
remote_count = sum(1 for l in listings_all if l.is_remote)
print(f"De las cuales, remote: {remote_count}")

# Top 10 tags más comunes
from collections import Counter
all_tags = [tag for l in listings_all for tag in l.tags]
top_tags = Counter(all_tags).most_common(10)
print(f"\nTop 10 tags:")
for tag, count in top_tags:
    print(f"  {tag}: {count}")

In [ ]:
import importlib
import hansel.sources
import hansel.sources.adzuna
importlib.reload(hansel.sources.adzuna)
importlib.reload(hansel.sources)

from dotenv import load_dotenv
load_dotenv()  # por si el kernel es nuevo

from hansel.sources import AdzunaAdapter

adapter = AdzunaAdapter()  # Suiza por defecto
print(f"Adapter: {adapter}\n")

# Búsqueda real: Python en Suiza
listings = await adapter.search(
    keywords="Junior Python Developer",
    location="Zurich",
    limit=5,
)

print(f"Encontradas {len(listings)} ofertas\n")
for l in listings:
    salary_info = ""
    if l.salary_min or l.salary_max:
        salary_info = f"  💰 {l.salary_min}-{l.salary_max} {l.salary_currency}"
    
    print(f"📌 {l.title}")
    print(f"   🏢 {l.company}")
    print(f"   📍 {l.location}  (remote: {l.is_remote})")
    print(f"   📅 Posted: {l.posted_at}")
    print(f"   🏷  {l.tags}")
    if salary_info:
        print(salary_info)
    print(f"   📝 {l.description[:150]}...")
    print()

In [ ]:
# Probar sin location para ver toda Suiza
listings_ch = await adapter.search(
    keywords="Junior Data Engineer",
    location=None,
    limit=5,
)

print(f"Encontradas {len(listings_ch)} ofertas Junior Data Engineer en toda Suiza\n")
for l in listings_ch:
    print(f"📌 {l.title}  @  {l.company}")
    print(f"   📍 {l.location}")
    print()

# Y cuántas de las primeras 20 son de Aarau/Aargau/Zurich
listings_sample = await adapter.search(
    keywords="Python",
    location=None,
    limit=20,
)

zones_of_interest = ["aargau", "aarau", "zurich", "zürich", "basel"]
relevant = [
    l for l in listings_sample
    if any(z in (l.location or "").lower() for z in zones_of_interest)
]
print(f"\nDe 20 ofertas 'Python' en Suiza, {len(relevant)} están cerca de Aarau/Zurich/Basel:")
for l in relevant:
    print(f"  • {l.title}  @  {l.location}")

In [ ]:
# Debug: misma búsqueda pero sin location
listings_debug1 = await adapter.search(
    keywords="Junior Python Developer",
    location=None,
    limit=5,
)
print(f"SIN location: {len(listings_debug1)} ofertas")
for l in listings_debug1:
    print(f"  • {l.title}  @  {l.location}")

# Con location más amplio
listings_debug2 = await adapter.search(
    keywords="Junior Python",
    location="Zurich",
    limit=5,
)
print(f"\n'Junior Python' en Zurich: {len(listings_debug2)} ofertas")
for l in listings_debug2:
    print(f"  • {l.title}  @  {l.location}")

# Con keywords más amplios en Zurich
listings_debug3 = await adapter.search(
    keywords="Junior Developer",
    location="Zurich",
    limit=5,
)
print(f"\n'Junior Developer' en Zurich: {len(listings_debug3)} ofertas")
for l in listings_debug3:
    print(f"  • {l.title}  @  {l.location}")

In [ ]:
# Recarga módulos
import importlib
import hansel.sources
import hansel.sources.orchestrator
importlib.reload(hansel.sources.orchestrator)
importlib.reload(hansel.sources)

from hansel.sources import AdzunaAdapter, ArbeitnowAdapter, JobSearchOrchestrator

# 1. Crear adapters
adzuna = AdzunaAdapter(country="ch")
arbeitnow = ArbeitnowAdapter()

# 2. Crear orquestador
orchestrator = JobSearchOrchestrator(
    adapters=[adzuna, arbeitnow],
    per_query_limit=10,
)

# 3. Reutilizamos las queries que generó el QueryGenerator antes (strategy)
print("Queries a ejecutar:")
for q in strategy.queries:
    print(f"  • {q.keywords}  @  {q.location}")

print(f"\nTotal: {len(strategy.queries)} queries × 2 adapters = {len(strategy.queries) * 2} llamadas HTTP en paralelo\n")

# 4. ¡Lanzamos!
import time
start = time.time()
listings = await orchestrator.search_all(strategy.queries)
elapsed = time.time() - start

print(f"⏱  Tardó: {elapsed:.1f}s")
print(f"📊 {len(listings)} ofertas únicas encontradas\n")

# 5. Breakdown por fuente
from collections import Counter
by_source = Counter(l.source.value for l in listings)
print("Por fuente:")
for source, count in by_source.most_common():
    print(f"  {source}: {count}")

# 6. Muestra las 5 más recientes
print("\n5 más recientes:")
sorted_by_date = sorted(
    [l for l in listings if l.posted_at],
    key=lambda l: l.posted_at,
    reverse=True,
)[:5]
for l in sorted_by_date:
    print(f"  📌 {l.title}")
    print(f"     🏢 {l.company}  📍 {l.location}")
    print(f"     📅 {l.posted_at.date()}  |  {l.source.value}")
    print()

In [ ]:
import time
from dotenv import load_dotenv
load_dotenv()

from hansel.sources import AdzunaAdapter, ArbeitnowAdapter, JobSearchOrchestrator

# Creamos instancias limpias
adzuna = AdzunaAdapter(country="ch")
arbeitnow = ArbeitnowAdapter()

print(f"Adzuna rate limiter interval: {adzuna._rate_limiter._min_interval}s")
print(f"Adzuna cache size: {adzuna._cache._cache.maxsize}")

In [ ]:
from hansel.search.schemas import JobQuery

queries = [
    JobQuery(keywords="Junior Backend Developer", location="Switzerland", rationale="..."),
    JobQuery(keywords="Junior Data Engineer", location="remote", rationale="..."),
    JobQuery(keywords="Junior Python Developer", location="Aarau", rationale="..."),
    JobQuery(keywords="Junior Big Data Engineer", location="Zurich", rationale="..."),
    JobQuery(keywords="Junior Software Developer", location="remote Europe", rationale="..."),
]

for q in queries:
    print(f"  • {q.keywords}  @  {q.location}")

In [ ]:
import logging

# Mostrar logs del rate limiter y retries
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s: %(message)s")

orchestrator = JobSearchOrchestrator(
    adapters=[adzuna, arbeitnow],
    per_query_limit=10,
)

print(f"Lanzando {len(queries)} queries × 2 adapters = {len(queries) * 2} llamadas...\n")

start = time.time()
listings = await orchestrator.search_all(queries)
elapsed = time.time() - start

print(f"\n⏱  Tardó: {elapsed:.1f}s")
print(f"📊 {len(listings)} ofertas únicas encontradas\n")

from collections import Counter
by_source = Counter(l.source.value for l in listings)
print("Por fuente:")
for source, count in by_source.most_common():
    print(f"  {source}: {count}")

print("\n5 más recientes:")
sorted_by_date = sorted(
    [l for l in listings if l.posted_at],
    key=lambda l: l.posted_at,
    reverse=True,
)[:5]
for l in sorted_by_date:
    print(f"  📌 {l.title}")
    print(f"     🏢 {l.company}  📍 {l.location}")
    print(f"     📅 {l.posted_at.date()}  |  {l.source.value}")

In [ ]:
import time
import logging
from dotenv import load_dotenv
load_dotenv()

# Logs informativos pero no de debug
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s")

from hansel.cv import CVExtractor, Seniority
from hansel.search import QueryGenerator
from hansel.sources import AdzunaAdapter, ArbeitnowAdapter, JobSearchOrchestrator

# Volvemos a necesitar el perfil - rápido
extractor = CVExtractor()
print("Extrayendo CV (2 min)...")
profile = extractor.extract_from_file(
    "./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.JUNIOR,
)
print(f"✓ Perfil extraído: {profile.full_name}")

In [ ]:
qg = QueryGenerator()
print("Generando queries (1 min)...")
strategy = qg.generate(
    profile=profile,
    preferred_location="Switzerland",
    remote_ok=True,
)
print("Queries generadas:")
for q in strategy.queries:
    print(f"  • {q.keywords}  @  {q.location}")

In [ ]:
adzuna = AdzunaAdapter(country="ch")
arbeitnow = ArbeitnowAdapter()
orchestrator = JobSearchOrchestrator(adapters=[adzuna, arbeitnow], per_query_limit=10)

print(f"\nLanzando orquestador con {len(strategy.queries)} queries...\n")
start = time.time()
listings = await orchestrator.search_all(strategy.queries)
elapsed = time.time() - start

print(f"\n⏱  Tardó: {elapsed:.1f}s")
print(f"📊 {len(listings)} ofertas únicas\n")

from collections import Counter
by_source = Counter(l.source.value for l in listings)
print("Por fuente:")
for source, count in by_source.most_common():
    print(f"  {source}: {count}")

# Top 10 más recientes
print("\n10 más recientes:")
sorted_by_date = sorted(
    [l for l in listings if l.posted_at],
    key=lambda l: l.posted_at,
    reverse=True,
)[:10]
for l in sorted_by_date:
    print(f"  📌 {l.title[:70]}")
    print(f"     🏢 {l.company}  📍 {l.location}  |  {l.source.value}")

In [ ]:
# Limpiamos el cache para forzar llamada fresca
adzuna._cache._cache.clear()

# Llamada directa con location="Switzerland"
result = await adzuna.search(
    keywords="Backend Developer", 
    location="Switzerland", 
    limit=10,
)
print(f"Resultados directos: {len(result)}")
for r in result[:5]:
    print(f"  • {r.title}  @  {r.location}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import time
from hansel.cv import CVExtractor, Seniority
from hansel.search import QueryGenerator
from hansel.sources import AdzunaAdapter, ArbeitnowAdapter, JobSearchOrchestrator
from hansel.matcher import (
    EmbeddingScorer,
    detect_title_seniority,
    filter_by_seniority_strict,
    cv_to_text,
    listing_to_text,
)

# Extraer CV (ya sabes que tarda)
print("Extrayendo CV...")
extractor = CVExtractor()
profile = extractor.extract_from_file(
    "./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.JUNIOR,
)
print(f"✓ Perfil: {profile.full_name}")

In [ ]:
print("Generando queries y buscando ofertas...")
qg = QueryGenerator()
strategy = qg.generate(profile, preferred_location="Switzerland", remote_ok=True)

adzuna = AdzunaAdapter(country="ch")
arbeitnow = ArbeitnowAdapter()
orchestrator = JobSearchOrchestrator(adapters=[adzuna, arbeitnow], per_query_limit=10)

listings = await orchestrator.search_all(strategy.queries)
print(f"✓ {len(listings)} ofertas encontradas")

In [ ]:
# Ver qué seniority detecta cada título
print("\nDetección de seniority por título:\n")
for l in listings[:15]:
    detected = detect_title_seniority(l.title)
    print(f"  {l.title[:60]:<60} → {detected.value if detected else 'unknown'}")

In [ ]:
# Filtrar las incompatibles con JUNIOR
print(f"\nAntes: {len(listings)} ofertas")
filtered = filter_by_seniority_strict(listings, Seniority.JUNIOR)
print(f"Después de filtrar por JUNIOR strict: {len(filtered)} ofertas")

# Ver las descartadas
descartadas = [l for l in listings if l not in filtered]
print(f"\nDescartadas ({len(descartadas)}):")
for l in descartadas:
    detected = detect_title_seniority(l.title)
    print(f"  ❌ {l.title[:55]:<55} ({detected.value if detected else '?'})")

In [ ]:
print("Probando embeddings v3 (solo títulos, sin description)...\n")

# Ver el texto nuevo primero
from hansel.matcher.embeddings import cv_to_text, listing_to_text

print("CV TEXT:")
print(f"  {cv_to_text(profile)}\n")

print("LISTING TEXTS (primeros 3):")
for l in filtered[:3]:
    print(f"  → {listing_to_text(l)}")

# Re-embed
scorer = EmbeddingScorer()
import time
start = time.time()
scored = await scorer.score_listings(profile, filtered)
elapsed = time.time() - start
print(f"\n⏱  {elapsed:.1f}s")

scored_sorted = sorted(scored, key=lambda x: x[1], reverse=True)

print("\nTop 10:")
for listing, score in scored_sorted[:10]:
    print(f"  {score:.3f}  │  {listing.title[:55]:<55} @ {listing.company}")

print("\nBottom 5:")
for listing, score in scored_sorted[-5:]:
    print(f"  {score:.3f}  │  {listing.title[:55]:<55} @ {listing.company}")

scores_only = [s for _, s in scored]
print(f"\nRango: {min(scores_only):.3f} - {max(scores_only):.3f} (spread: {max(scores_only)-min(scores_only):.3f})")

# Dónde queda la oferta perfecta
for rank, (l, s) in enumerate(scored_sorted, 1):
    if "Junior/Mid Python" in l.title:
        print(f"\n🎯 'Junior/Mid Python Developer' está en el puesto #{rank} con score {s:.3f}")
        break

In [ ]:
# Ver exactamente qué texto estamos embeddando
from hansel.matcher.embeddings import cv_to_text, listing_to_text

print("=" * 70)
print("CV TEXT (lo que se embeddea como 'candidato'):")
print("=" * 70)
print(cv_to_text(profile))

# Comparar dos listings: uno obvio match, otro obvio no-match
obvious_match = next(l for l in filtered if "Junior/Mid Python" in l.title)
random_one = next(l for l in filtered if "Mechatronics" in l.title)

print("\n" + "=" * 70)
print("LISTING TEXT (obvious MATCH - Junior/Mid Python):")
print("=" * 70)
print(listing_to_text(obvious_match))

print("\n" + "=" * 70)
print("LISTING TEXT (questionable - Mechatronics):")
print("=" * 70)
print(listing_to_text(random_one))

In [ ]:
# Reinicia kernel si es necesario, luego:
import importlib
import hansel.matcher
importlib.reload(hansel.matcher)

from hansel.matcher import LLMReranker

reranker = LLMReranker()

# Escogemos la oferta "Junior/Mid Python Developer" (nuestra oferta perfecta)
target = next(l for l in filtered if "Junior/Mid Python" in l.title)

print(f"Scoring: {target.title}")
print(f"Company: {target.company}")
print(f"Description preview: {target.description[:200]}...\n")

import time
start = time.time()
score = await reranker.score(profile, target)
elapsed = time.time() - start

print(f"⏱  {elapsed:.1f}s\n")

if score:
    print(f"Overall:       {score.overall:.2f}")
    print(f"Skills match:  {score.skills_match:.2f}")
    print(f"Seniority fit: {score.seniority_fit:.2f}")
    print(f"\nRationale:\n{score.rationale}")
else:
    print("❌ Scoring failed")

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s")

from hansel.cv import CVExtractor, Seniority
from hansel.search import QueryGenerator
from hansel.sources import AdzunaAdapter, ArbeitnowAdapter, JobSearchOrchestrator
from hansel.matcher import JobMatcher, SeniorityMode

# Pipeline completo
print("1️⃣  Extrayendo CV...")
extractor = CVExtractor()
profile = extractor.extract_from_file(
    "./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.JUNIOR,
)
print(f"✓ {profile.full_name}\n")

print("2️⃣  Generando queries...")
qg = QueryGenerator()
strategy = qg.generate(profile, preferred_location="Switzerland", remote_ok=True)
print(f"✓ {len(strategy.queries)} queries\n")

print("3️⃣  Buscando ofertas...")
orchestrator = JobSearchOrchestrator(
    adapters=[AdzunaAdapter(country="ch"), ArbeitnowAdapter()],
    per_query_limit=10,
)
listings = await orchestrator.search_all(strategy.queries)
print(f"✓ {len(listings)} ofertas encontradas\n")

print("4️⃣  Ranking con Matcher (~8-12 min)...")
print("    (filtro seniority + embeddings + LLM rerank top 10)\n")
matcher = JobMatcher(
    rerank_top_n=10,
    seniority_mode=SeniorityMode.STRICT,
)
start = time.time()
ranked = await matcher.rank(profile, listings)
elapsed = time.time() - start
print(f"\n✓ Matcher terminó en {elapsed/60:.1f} min\n")

1️⃣  Extrayendo CV...


2026-04-22 09:56:03,578 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


✓ MARIA LOPEZ GARCIA

2️⃣  Generando queries...


2026-04-22 09:58:13,055 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 09:58:45,744 hansel.sources.orchestrator: Orchestrating 4 queries across 2 adapters (8 total calls)


✓ 4 queries

3️⃣  Buscando ofertas...


2026-04-22 09:58:46,253 httpx: HTTP Request: GET https://www.arbeitnow.com/api/job-board-api "HTTP/1.1 200 OK"
2026-04-22 09:58:46,678 httpx: HTTP Request: GET https://www.arbeitnow.com/api/job-board-api "HTTP/1.1 200 OK"
2026-04-22 09:58:46,859 hansel.sources.arbeitnow: Arbeitnow returned 100 raw items
2026-04-22 09:58:46,860 hansel.sources.arbeitnow: Arbeitnow matched 0 items for keywords='Backend Developer' location='Switzerland'
2026-04-22 09:58:46,861 hansel.sources.arbeitnow: Arbeitnow returned 100 raw items
2026-04-22 09:58:46,862 hansel.sources.arbeitnow: Arbeitnow matched 0 items for keywords='Data Engineer' location='remote'
2026-04-22 09:58:46,882 httpx: HTTP Request: GET https://api.adzuna.com/v1/api/jobs/ch/search/1?app_id=ccd54d18&app_key=09fc5bc6b610bd95486888caadb08663&results_per_page=10&what=Backend+Developer&content-type=application%2Fjson "HTTP/1.1 200 OK"
2026-04-22 09:58:46,884 hansel.sources.adzuna: Adzuna: 10 items of 30 total (keywords='Backend Developer', loca

✓ 27 ofertas encontradas

4️⃣  Ranking con Matcher (~8-12 min)...
    (filtro seniority + embeddings + LLM rerank top 10)



2026-04-22 09:58:50,916 httpx: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-04-22 09:58:51,345 httpx: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
2026-04-22 09:58:51,350 hansel.matcher.embeddings: Embedding scoring complete
2026-04-22 09:58:51,350 hansel.matcher.matcher: Reranking top 10 listings with LLM (remaining 9 keep embedding-only scores)
2026-04-22 09:58:51,351 hansel.matcher.reranker: LLM-reranking 10 listings...
2026-04-22 09:59:15,951 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 09:59:36,146 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:00:00,982 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:00:23,366 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:00:49,947 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:01:0


✓ Matcher terminó en 4.2 min



In [2]:
print(f"📊 {len(ranked)} ofertas ranqueadas\n")
print("=" * 80)
print("TOP 5 MATCHES:")
print("=" * 80)

for i, sl in enumerate(ranked[:5], 1):
    print(f"\n#{i}  score {sl.final_score:.2f}  |  {sl.listing.title}")
    print(f"    🏢 {sl.listing.company}  📍 {sl.listing.location}")
    print(f"    embedding: {sl.embedding_score:.2f}", end="")
    if sl.llm_score:
        print(f"  |  LLM overall: {sl.llm_score.overall:.2f}  skills: {sl.llm_score.skills_match:.2f}  seniority: {sl.llm_score.seniority_fit:.2f}")
        print(f"    💬 {sl.llm_score.rationale}")
    else:
        print("  (no LLM rerank)")
    print(f"    🔗 {sl.listing.url}")

print("\n" + "=" * 80)
print("POSICIONES 6-10 (reranked, pero menores scores):")
print("=" * 80)
for i, sl in enumerate(ranked[5:10], 6):
    print(f"#{i}  score {sl.final_score:.2f}  |  {sl.listing.title} @ {sl.listing.company}")
    if sl.llm_score:
        print(f"    💬 {sl.llm_score.rationale[:150]}...")

📊 19 ofertas ranqueadas

TOP 5 MATCHES:

#1  score 0.54  |  Visual AI Data Engineering - internship
    🏢 ANYbotics  📍 Schweiz
    embedding: 0.63  |  LLM overall: 0.50  skills: 0.60  seniority: 0.50
    💬 Candidate has relevant backend and data engineering skills but lacks experience in AI and robotics, which are key for the role. Good fit for a junior position with potential for growth.
    🔗 https://www.adzuna.ch/details/5681714334?utm_medium=api&utm_source=ccd54d18

#2  score 0.53  |  Backend Developer (VoIP)
    🏢 Alohi SA  📍 Kanton Genf, Schweiz
    embedding: 0.66  |  LLM overall: 0.65  skills: 0.75  seniority: 1.00
    💬 Candidate has strong backend development skills and experience in building APIs, which aligns well with the role. The main gap is the specific VoIP focus, but this can be learned on the job given the candidate's background.
    🔗 https://www.adzuna.ch/details/5576766760?utm_medium=api&utm_source=ccd54d18

#3  score 0.53  |  Data Engineer (a) in Zürich
    🏢 Axe

In [3]:
# ¿Cuántas ofertas tienen llm_score NO-None en el ranking?
con_llm = [sl for sl in ranked if sl.llm_score is not None]
sin_llm = [sl for sl in ranked if sl.llm_score is None]

print(f"Con LLM score: {len(con_llm)}")
print(f"Sin LLM score: {len(sin_llm)}")
print(f"Total: {len(ranked)}\n")

# ¿Qué posiciones tienen rerank?
print("Ofertas CON rerank LLM:")
for sl in con_llm:
    idx = ranked.index(sl) + 1
    print(f"  #{idx}  emb={sl.embedding_score:.2f}  llm_overall={sl.llm_score.overall:.2f}  |  {sl.listing.title[:50]}")

print("\nOfertas SIN rerank LLM (embedding only):")
for sl in sin_llm:
    idx = ranked.index(sl) + 1
    print(f"  #{idx}  emb={sl.embedding_score:.2f}  |  {sl.listing.title[:50]}")

Con LLM score: 10
Sin LLM score: 9
Total: 19

Ofertas CON rerank LLM:
  #1  emb=0.63  llm_overall=0.50  |  Visual AI Data Engineering - internship
  #2  emb=0.66  llm_overall=0.65  |  Backend Developer (VoIP)
  #3  emb=0.64  llm_overall=0.65  |  Data Engineer (a) in Zürich
  #4  emb=0.62  llm_overall=0.65  |  Data Engineer in Zürich
  #7  emb=0.63  llm_overall=0.45  |  Internship : SOAR Automation Engineer
  #8  emb=0.69  llm_overall=0.60  |  Data Engineer
  #9  emb=0.63  llm_overall=0.50  |  Professional Web Developer (Python/JavaScript, 80%
  #10  emb=0.69  llm_overall=0.45  |  Consultant Data Engineer
  #15  emb=0.65  llm_overall=0.45  |  Test System Software Engineer
  #19  emb=0.65  llm_overall=0.15  |  Mechatronics Engineer

Ofertas SIN rerank LLM (embedding only):
  #5  emb=0.60  |  Junior Software Engineer – Data Analysis & Automat
  #6  emb=0.58  |  Junior/Mid Python Developer
  #11  emb=0.62  |  Forward Deployed Engineer - German Speaking
  #12  emb=0.61  |  Fullstack Front- 

In [4]:
import importlib
import hansel.email_gen.schemas
import hansel.email_gen.validator
import hansel.email_gen.generator
import hansel.email_gen
importlib.reload(hansel.email_gen.schemas)
importlib.reload(hansel.email_gen.validator)
importlib.reload(hansel.email_gen.generator)
importlib.reload(hansel.email_gen)

from hansel.email_gen import EmailGenerator, EmailLanguage, GeneratedEmail, EmailGenerationSkipped

# enable_fact_check=True por defecto, lo ponemos explícito para claridad
generator = EmailGenerator(enable_fact_check=True)

for i, sl in enumerate(ranked[:3], 1):
    print(f"\n{'═'*70}")
    print(f"EMAIL #{i}: {sl.listing.title} @ {sl.listing.company}  (score {sl.final_score:.2f})")
    print('═'*70)
    
    result = await generator.generate(
        profile=profile,
        scored_listing=sl,
        language=EmailLanguage.ENGLISH,
        word_count_target=150,
    )
    
    if isinstance(result, EmailGenerationSkipped):
        print(f"⚠️  SKIPPED:\n{result.reason}")
    else:
        print(f"Subject: {result.subject}\n")
        print(result.body)
        print(f"\n[{result.word_count} words]")

2026-04-22 10:02:59,624 hansel.email_gen.generator: Drafting email for 'Visual AI Data Engineering - internship'...



══════════════════════════════════════════════════════════════════════
EMAIL #1: Visual AI Data Engineering - internship @ ANYbotics  (score 0.54)
══════════════════════════════════════════════════════════════════════


2026-04-22 10:03:57,218 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:04:19,529 hansel.email_gen.generator: Critiquing and refining...
2026-04-22 10:04:45,455 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:05:18,756 hansel.email_gen.generator: Critique rejected (79 words, hallucinations: ['Subject line is empty']). Keeping draft.
2026-04-22 10:05:18,757 hansel.email_gen.generator: Fact-checking generated email...
2026-04-22 10:05:48,149 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:05:52,656 hansel.email_gen.generator: Drafting email for 'Backend Developer (VoIP)'...


Subject: Junior Python Developer application — 1.5 years REST API experience

Your posting emphasizes building data pipelines for production ML—an area I've been focused on for the past year. At TechStart SL, I built robust REST APIs and data pipelines using FastAPI and PostgreSQL, honing my skills in clean code and testing. However, I recognize a gap in AI and robotics experience, which is key for this role. Would a 20-minute call next week be possible?

[65 words]

══════════════════════════════════════════════════════════════════════
EMAIL #2: Backend Developer (VoIP) @ Alohi SA  (score 0.53)
══════════════════════════════════════════════════════════════════════


2026-04-22 10:06:50,022 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:07:18,667 hansel.email_gen.generator: Critiquing and refining...
2026-04-22 10:07:45,719 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:08:24,977 hansel.email_gen.generator: Critique rejected (93 words, hallucinations: ['Subject line is empty']). Keeping draft.
2026-04-22 10:08:24,978 hansel.email_gen.generator: Fact-checking generated email...
2026-04-22 10:08:57,499 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:09:00,228 hansel.email_gen.generator: Drafting email for 'Data Engineer (a) in Zürich'...


Subject: Backend Developer (VoIP) — 1.5 years REST API and data pipeline experience

Your posting emphasizes building data pipelines for production ML—an area I've been focused on for the past year. At TechStart SL, I built robust REST APIs and data pipelines, honing my skills with Python, FastAPI, and PostgreSQL. My recent specialization in AI and Big Data aligns well with Alohi's mission to merge state-of-the-art technologies with user experience. While my current experience doesn't specifically include VoIP, I am eager to learn and contribute to your team. Would a 20-minute call next week be possible?

[83 words]

══════════════════════════════════════════════════════════════════════
EMAIL #3: Data Engineer (a) in Zürich @ Axept Business Software AG  (score 0.53)
══════════════════════════════════════════════════════════════════════


2026-04-22 10:09:59,659 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:10:26,070 hansel.email_gen.generator: Critiquing and refining...
2026-04-22 10:10:51,140 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-22 10:11:23,308 hansel.email_gen.generator: Critique rejected (76 words, hallucinations: ['Subject line is empty']). Keeping draft.
2026-04-22 10:11:23,308 hansel.email_gen.generator: Fact-checking generated email...
2026-04-22 10:11:56,512 httpx: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Subject: Junior Backend Developer Application — 1.5 years REST API and Data Pipeline Experience

Your posting emphasizes building data pipelines for production ML—an area I've been focused on for the past year. At TechStart SL, I built robust REST APIs and data pipelines using Python and FastAPI, ensuring clean code and reliable systems. My recent specialization in AI and Big Data aligns well with your innovative projects at Axept Business Software AG. However, my preference for remote roles might be a gap given this Zürich-based position. Would a 20-minute call next week be possible?

[80 words]


In [2]:
from dotenv import load_dotenv
load_dotenv()

import logging
logging.basicConfig(level=logging.WARNING)  # Menos verboso para verlo limpio

from hansel import HanselAgent
from hansel.cv.schemas import Seniority
from hansel.email_gen import GeneratedEmail, EmailGenerationSkipped

# Callback de progreso simple
def progress(step: str, detail: str) -> None:
    print(f"  → [{step}] {detail}")

agent = HanselAgent(progress_callback=progress)

import time
start = time.time()
result = await agent.find_jobs(
    cv_path="./tests/fixtures/cv_junior_tech.md",
    seniority=Seniority.JUNIOR,
    preferred_location="Switzerland",
    generate_emails=True,
    emails_top_n=3,
)
elapsed = time.time() - start

print(f"\n⏱  Total: {elapsed/60:.1f} min")
print(f"Success: {result.success}")
print(f"Ranked listings: {len(result.ranked_listings)}")
print(f"Emails: {len(result.emails)}")

  → [extract_cv] Parsing ./tests/fixtures/cv_junior_tech.md
  → [extract_cv] Extracted 23 skills, 2 experiences, 3 target roles
  → [generate_queries] Generating diverse search queries
  → [generate_queries] Generated 4 queries
  → [search] Querying 2 sources in parallel
  → [search] Found 46 unique listings
  → [rank] Ranking 46 listings (embeddings + LLM rerank)
  → [rank] Ranked 35 listings. Top score: 0.54
  → [generate_emails] Generating emails for top 3 listings
  → [generate_emails] Email 1/3: Data Engineer
  → [generate_emails] Email 2/3: Data Engineer


  → [generate_emails] Email 3/3: Data Engineer


  → [generate_emails] Generated 3/3 emails successfully
  → [complete] Pipeline finished successfully

⏱  Total: 18.5 min
Success: True
Ranked listings: 35
Emails: 3


In [3]:
print(f"\n{'='*70}\nTOP 5 MATCHES\n{'='*70}")
for i, sl in enumerate(result.ranked_listings[:5], 1):
    print(f"\n#{i}  score {sl.final_score:.2f}  |  {sl.listing.title}")
    print(f"   🏢 {sl.listing.company}  📍 {sl.listing.location}")
    if sl.llm_score:
        print(f"   💬 {sl.llm_score.rationale}")

print(f"\n{'='*70}\nEMAILS\n{'='*70}")
for i, email in enumerate(result.emails, 1):
    if isinstance(email, GeneratedEmail):
        print(f"\n--- Email {i} ({email.word_count} words) ---")
        print(f"Subject: {email.subject}\n")
        print(email.body)
    else:
        print(f"\n--- Email {i}: SKIPPED ---")
        print(email.reason)


TOP 5 MATCHES

#1  score 0.54  |  Data Engineer
   🏢 Destinus  📍 Schweiz
   💬 Strong skills in backend and data engineering align well with the role. However, the candidate's recent specialization in AI and Big Data may not be directly relevant to the immediate needs of this position.

#2  score 0.54  |  Data Engineer
   🏢 Garda Capital Partners  📍 Schweiz
   💬 Candidate has strong backend and data engineering skills, including Python and relevant tools. The role is a stretch for the candidate's experience but not overly so. Gap in direct experience with some required technologies like FastAPI and specific database systems.

#3  score 0.54  |  Data Engineer
   🏢 Verisign  📍 Schweiz
   💬 The candidate has strong backend and data engineering skills relevant to the role, including Python, FastAPI, and PostgreSQL. The seniority fit is good as a junior candidate for a junior role. However, there's limited experience with big data tools which could be a gap.

#4  score 0.53  |  Backend Deve